# Code mẫu — Gradient boosting

Gradient boosting dựng model từng bước. Mỗi cây mới học cách sửa phần còn sai của những cây trước. Ví dụ đầu nhìn tận mắt phần dư và tác động của `learning_rate`. Hai ví dụ sau dùng validation để chọn số vòng và tốc độ học, và chỉ mở test khi đã chọn xong.

## Ví dụ 1: Một stump sửa phần dư

Bốn target là `[1, 1, 3, 3]`. Dự đoán ban đầu là trung bình, bằng 2. Như vậy hai mẫu đầu cần hạ xuống, còn hai mẫu sau cần nâng lên. Trước khi gọi thư viện, ta tự tính hai giá trị sửa cho phép chia nằm giữa `x = 1` và `x = 2`.

In [ ]:
import numpy as np

x = np.array([[0.], [1.], [2.], [3.]])
y = np.array([1., 1., 3., 3.])
initial = np.full(len(y), y.mean())
residual_to_fix = y - initial
left = x[:, 0] <= 1.
manual_left = residual_to_fix[left].mean()
manual_right = residual_to_fix[~left].mean()
print("F0:", initial.tolist())
print("phần dư y - F0:", residual_to_fix.tolist())
print("bản sửa trái / phải:", manual_left, manual_right)
assert np.allclose(initial, 2.)
assert np.isclose(manual_left, -1.) and np.isclose(manual_right, 1.)

F0: [2.0, 2.0, 2.0, 2.0]
phần dư y - F0: [-1.0, -1.0, 1.0, 1.0]
bản sửa trái / phải: -1.0 1.0


Với loss bình phương, cây tiếp theo học đúng phần dư `y − dự đoán`. Dấu của phần dư cho biết hướng sửa: âm thì hạ dự đoán xuống, dương thì nâng lên. Nhánh trái cần sửa −1, nhánh phải cần sửa +1.

Giờ để thư viện làm việc ấy qua ba vòng, với hai learning rate khác nhau. Mỗi vòng fit một stump, tức cây chỉ có một phép chia, lên phần dư hiện tại, rồi cộng thêm `learning_rate` lần dự đoán của stump.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

for learning_rate in [1.0, 0.5]:
    prediction = np.full(len(y), y.mean())
    for stage in range(3):
        correction = DecisionTreeRegressor(max_depth=1, random_state=0)
        correction.fit(x, y - prediction)
        prediction += learning_rate * correction.predict(x)
        mse = np.mean((prediction - y) ** 2)
        print("lr =", learning_rate, "vòng", stage + 1,
              "dự đoán:", prediction.round(3).tolist(),
              "MSE:", round(float(mse), 4))

lr = 1.0 vòng 1 dự đoán: [1.0, 1.0, 3.0, 3.0] MSE: 0.0
lr = 1.0 vòng 2 dự đoán: [1.0, 1.0, 3.0, 3.0] MSE: 0.0
lr = 1.0 vòng 3 dự đoán: [1.0, 1.0, 3.0, 3.0] MSE: 0.0
lr = 0.5 vòng 1 dự đoán: [1.5, 1.5, 2.5, 2.5] MSE: 0.25
lr = 0.5 vòng 2 dự đoán: [1.25, 1.25, 2.75, 2.75] MSE: 0.0625
lr = 0.5 vòng 3 dự đoán: [1.125, 1.125, 2.875, 2.875] MSE: 0.0156


Với learning rate 1, stump đầu tiên sửa trọn vẹn, và dữ liệu đã khớp hoàn toàn ngay vòng 1. Các vòng sau không còn gì để sửa. Với learning rate 0.5, mỗi vòng chỉ đi nửa quãng đường còn lại, nên phần dư cứ thế giảm một nửa: 0.5, rồi 0.25, rồi 0.125. Dữ liệu sạch như thế này chỉ để kiểm dấu và cơ chế. Với dữ liệu thật có nhiễu, thêm cây có thể bắt đầu học thuộc nhiễu, nên cần validation để biết lúc nào dừng.

## Ví dụ 2: Chọn số vòng bằng validation sau từng vòng

Ta chia dữ liệu hai lần để có ba tập: train, validation và test. Fit một model 180 cây trên train. Hàm `staged_predict` cho dự đoán sau từng vòng, nên ta đo được MSE validation sau vòng 1, vòng 2, cho tới vòng 180, mà không phải fit lại. Chọn vòng có MSE thấp nhất, fit lại đúng số vòng đó, rồi mới chấm test. Validation đã được dùng để chọn, nên nó không còn là đánh giá độc lập.

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(31)
x = np.linspace(0., 8., 180)
y = np.sin(1.7 * x) + rng.normal(0., 0.22, len(x))
X = x[:, None]
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=11
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=8
)

params = dict(max_depth=3, learning_rate=0.1, random_state=0)
full = GradientBoostingRegressor(n_estimators=180, **params).fit(X_train, y_train)
valid_losses = np.array([
    mean_squared_error(y_valid, prediction)
    for prediction in full.staged_predict(X_valid)
])
best_round = int(np.argmin(valid_losses)) + 1
print("vòng tốt nhất:", best_round)
print("MSE valid vòng 1 / tốt nhất / cuối:",
      round(float(valid_losses[0]), 3),
      round(float(valid_losses[best_round - 1]), 3),
      round(float(valid_losses[-1]), 3))

selected = GradientBoostingRegressor(n_estimators=best_round, **params).fit(
    X_train, y_train
)
test_mse = mean_squared_error(y_test, selected.predict(X_test))
print("MSE test sau khi khóa số vòng:", round(float(test_mse), 3))
assert 1 <= best_round <= 180
assert np.isfinite(test_mse)

vòng tốt nhất: 62
MSE valid vòng 1 / tốt nhất / cuối: 0.474 0.055 0.07
MSE test sau khi khóa số vòng: 0.065


Dữ liệu là một đường sin có nhiễu. MSE validation giảm từ 0.474 ở vòng 1 xuống 0.055 ở vòng 62, rồi tăng trở lại 0.070 ở vòng 180. Sau vòng 62, các cây mới bắt đầu học nhiễu của train, và đây chính là lý do để dừng sớm. Nếu vòng tốt nhất rơi đúng vào vòng 180, ta chưa có lý do để dừng sớm, và nên thử giới hạn số vòng lớn hơn.

Model `selected` vẫn chỉ học từ train. Test chỉ được chấm sau khi số vòng đã chốt, nên 0.065 là một đánh giá độc lập. Nếu dùng chính test để chọn số vòng, con số này sẽ đẹp hơn thực tế.

## Ví dụ 3: So hai tốc độ học với cùng 180 cây

Learning rate nhỏ thường cần nhiều cây hơn mới đạt cùng mức lỗi. Ta giữ cố định 180 cây, độ sâu, seed và cách chia, chỉ đổi learning rate, rồi so trên validation. Chọn xong, ta gộp train với validation để fit lại tốc độ đã chọn, rồi chấm test.

Dữ liệu là bộ Friedman do scikit-learn sinh ra. Target được tính bằng một công thức phi tuyến, có tích và bình phương của năm cột đầu, cộng thêm nhiễu.

In [ ]:
import numpy as np
from sklearn.datasets import make_friedman1
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

X, y = make_friedman1(n_samples=280, n_features=5, noise=1.2, random_state=8)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=10
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=2
)

validation_rows = []
for learning_rate in [0.20, 0.05]:
    model = GradientBoostingRegressor(
        n_estimators=180, max_depth=2, learning_rate=learning_rate,
        random_state=0
    ).fit(X_train, y_train)
    valid_mae = mean_absolute_error(y_valid, model.predict(X_valid))
    validation_rows.append((learning_rate, float(valid_mae)))
    print("lr =", learning_rate, "MAE valid:", round(float(valid_mae), 3))

best_lr, best_valid_mae = min(validation_rows, key=lambda row: row[1])
X_fit = np.vstack([X_train, X_valid])
y_fit = np.concatenate([y_train, y_valid])
chosen = GradientBoostingRegressor(
    n_estimators=180, max_depth=2, learning_rate=best_lr, random_state=0
).fit(X_fit, y_fit)
test_mae = mean_absolute_error(y_test, chosen.predict(X_test))
print("lr chọn:", best_lr,
      "valid đã chọn:", round(best_valid_mae, 3),
      "MAE test:", round(float(test_mae), 3))
print("3 cặp target / dự đoán:", np.column_stack([
    y_test[:3], chosen.predict(X_test[:3])
]).round(3).tolist())
assert best_lr in {0.05, 0.2}
assert np.isfinite(test_mae)

lr = 0.2 MAE valid: 1.49
lr = 0.05 MAE valid: 1.537
lr chọn: 0.2 valid đã chọn: 1.49 MAE test: 1.738
3 cặp target / dự đoán: [[16.698, 17.345], [17.377, 14.379], [17.473, 15.019]]


Với 180 cây, learning rate 0.2 thắng sát nút, MAE validation 1.49 so với 1.537. Chưa thể kết luận 0.05 kém hơn nói chung. Có thể nó chỉ cần thêm vòng. Muốn so công bằng, phải chọn số vòng riêng cho từng tốc độ, như cách làm ở ví dụ 2.

MAE test là 1.738, cao hơn con số 1.49 trên validation. Điều này hợp lý, vì 1.49 là điểm thắng sau một lần chọn, nên thường hơi lạc quan. Ba cặp cuối nhắc rằng MAE là trung bình: mẫu thứ hai bị đoán thấp khoảng 3 đơn vị, gần gấp đôi mức trung bình.